# HyperParameter Tuning using Optuna

What will we change this tune using optuna:

1. Number of hidden layers
2. Neurons per layer
3. Number of Epochs
4. Optimizer
5. Learning Rate
6. Batch size
7. Dropout rate
8. Weight decay (lambda)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device:{device}")

Using Device:cuda


In [3]:
df = pd.read_csv("fashion-mnist_train.csv")

In [4]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [7]:
import numpy as np
X = np.nan_to_num(X,nan=0.0)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [9]:
# performing the scailing
X_train = X_train/255.0
X_test = X_test/255.0

In [12]:
# Create a CustomDataset class
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):

    return len(self.features)

  def __getitem__(self,index):

    return self.features[index], self.labels[index]

In [13]:
# create train_dataset object
train_dataset = CustomDataset(X_train, y_train)

In [14]:
test_dataset = CustomDataset(X_test,y_test)

In [15]:
# create train and test loader object
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True) # pin_memory is simply used to speed up the training on GPU
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True) # while predicting, you usually dont wanna shuffle the data.

In [21]:
# Define a NN class for Optuna Hyperparameter tuning
class myNN(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer):

    super().__init__() # invoking the parent constructor

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(p=0.2))
      input_dim = neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers) # * means unpacking the layers

  def forward(self, X):

    return self.model(X)

In [22]:
# objective Function
def objective(trial):

  # Extract hyperparameter values for next trial from the search space
  num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step = 8)

  # model init
  input_dim = 784
  output_dim = 10

  model = myNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer)
  model.to(device)

  # params init
  epochs=10
  learning_rate = 0.1

  # optimizer selection
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

  # training loop
  for epoch in range(epochs):

    total_epoch_loss = 0 # setting up a variable in order to track the loss across each epoch

    for batch_features, batch_labels in train_loader:

      # move data to GPU
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # loss calculate
      loss = criterion(outputs, batch_labels)

      # back propagation
      optimizer.zero_grad() # clearing out gradients
      loss.backward()

      # update weights
      optimizer.step()

  # evaluation
  model.eval()
  # Evaluation code
  total = 0
  correct = 0

  with torch.no_grad():
      for batch_features, batch_labels in test_loader:
        # move data to GPU
          batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
          outputs = model(batch_features)

          _, predicted = torch.max(outputs, 1)  # Calculates class with max probability

          total += batch_labels.shape[0]      # Accumulate total samples

          correct += (predicted == batch_labels).sum().item()  # Count correct predictions

  accuracy = correct / total

  return accuracy

In [10]:
# Now we just need to create a study
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 33.8 MB/s eta 0:00:00


In [11]:
import optuna

study = optuna.create_study(direction='maximize')

[I 2026-04-02 20:45:46,314] A new study created in memory with name: no-name-34a62b55-2ba7-459f-b006-5e7aff3e1fd9


In [23]:
study.optimize(objective, n_trials=10)

[I 2026-04-02 20:49:36,128] Trial 1 finished with value: 0.86875 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 48}. Best is trial 1 with value: 0.86875.
[I 2026-04-02 20:53:46,721] Trial 2 finished with value: 0.8925 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 128}. Best is trial 2 with value: 0.8925.
[I 2026-04-02 20:56:58,947] Trial 3 finished with value: 0.8794166666666666 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 48}. Best is trial 2 with value: 0.8925.
[I 2026-04-02 20:59:39,559] Trial 4 finished with value: 0.86625 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 24}. Best is trial 2 with value: 0.8925.
[I 2026-04-02 21:01:46,623] Trial 5 finished with value: 0.87175 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 32}. Best is trial 2 with value: 0.8925.
[W 2026-04-02 21:01:52,510] Trial 6 failed with parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 128} because of the following error: Keybo

KeyboardInterrupt: 

In [ ]:
study.best_value

In [ ]:
study.best_params

The above code was just an experimennt.

Now let's add all the hyperparameter that needs tuning and then run the model.

In [24]:
# Define a NN class for Optuna Hyperparameter tuning
class myNN(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):

    super().__init__() # invoking the parent constructor

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(p=dropout_rate))
      input_dim = neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers) # * means unpacking the layers

  def forward(self, X):

    return self.model(X)

In [28]:
# objective Function
def objective(trial):

  # Extract hyperparameter values for next trial from the search space
  num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step = 8)
  epochs = trial.suggest_int("epochs", 10, 50, step = 10)
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
  dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical("batch_size", [16,32,64,128]) # batch_size is contained in dataloader code cell. But it's outside the objective function. so now we will need to write the code for dataloader inside the obj function
  optimizer_name = trial.suggest_categorical("optimizer", ['Adam','SGD','RMSprop']) # see the if else statement below
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

  # create train and test loader object
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)


  # model init
  input_dim = 784
  output_dim = 10

  model = myNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate) # we are sending the dropout_rate value as well cuz that's what's required by the myNN class above
  model.to(device)

  # optimizer selection
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  if optimizer_name == 'Adam':
    optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # training loop
  for epoch in range(epochs):

    total_epoch_loss = 0 # setting up a variable in order to track the loss across each epoch

    for batch_features, batch_labels in train_loader:

      # move data to GPU
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # loss calculate
      loss = criterion(outputs, batch_labels)

      # back propagation
      optimizer.zero_grad() # clearing out gradients
      loss.backward()

      # update weights
      optimizer.step()

  # evaluation
  model.eval()
  # Evaluation code
  total = 0
  correct = 0

  with torch.no_grad():
      for batch_features, batch_labels in test_loader:
        # move data to GPU
          batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
          outputs = model(batch_features)

          _, predicted = torch.max(outputs, 1)  # Calculates class with max probability

          total += batch_labels.shape[0]      # Accumulate total samples

          correct += (predicted == batch_labels).sum().item()  # Count correct predictions

  accuracy = correct / total

  return accuracy

In [26]:
import optuna

study = optuna.create_study(direction='maximize')

[I 2026-04-02 21:03:08,685] A new study created in memory with name: no-name-4f8b624a-6035-480f-99ac-1053a4050822


In [29]:
study.optimize(objective, n_trials=10)

[I 2026-04-02 21:05:00,496] Trial 1 finished with value: 0.61875 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 72, 'epochs': 20, 'learning_rate': 1.6656732881731052e-05, 'dropout_rate': 0.4, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 0.00044532720947367476}. Best is trial 1 with value: 0.61875.
[I 2026-04-02 21:05:30,840] Trial 2 finished with value: 0.3418333333333333 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 8, 'epochs': 20, 'learning_rate': 0.04524362402818301, 'dropout_rate': 0.4, 'batch_size': 128, 'optimizer': 'Adam', 'weight_decay': 0.0006797160648147428}. Best is trial 1 with value: 0.61875.
[I 2026-04-02 21:08:51,611] Trial 3 finished with value: 0.66525 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 16, 'epochs': 40, 'learning_rate': 0.015371613460824027, 'dropout_rate': 0.30000000000000004, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 1.1968371320890009e-05}. Best is trial 3 with value: 0.66525.
[

In [30]:
study.best_value

0.88475

In [31]:
study.best_params

{'num_hidden_layers': 5,
 'neurons_per_layer': 80,
 'epochs': 40,
 'learning_rate': 0.011235341215573591,
 'dropout_rate': 0.2,
 'batch_size': 64,
 'optimizer': 'Adam',
 'weight_decay': 2.2979578428655757e-05}

# To improve the accuracy further:

1. Increase number of trials
2. Increase the search space for the parameters in obj function

# Also make sure to use MLflow to track all the trials :)